# HackUDC 2026 — Inditex Fashion Retrieval (Improved)

**Objetivo:** maximizar Recall@15 recuperando hasta 15 productos por bundle.

**Prioridad real (de mayor a menor impacto):**
1. **Embeddings visuales + calidad de crops** (mayor impacto).
2. **Cobertura de candidatos en FAISS** (`K_PER_SEGMENT`, `IndexFlatIP`).
3. **Agregación multi-segmento** (pesos, bonus por intersección, whole-image bonus).
4. **Re-ranking fino** (PRF y texto) para mejoras incrementales.

**Arquitectura actual:**
- Marqo-FashionSigLIP (encoder principal, 512-dim)
- SegFormer B2 → crops por prenda
- TTA (original + flip + centre crop)
- FAISS IndexFlatIP (búsqueda exacta)
- Z-score + suma ponderada por segmento
- Bonus por intersección entre segmentos
- PRF (pseudo-relevance feedback)
- Soft category filtering (penalización, no exclusión dura)
- Text re-ranking (ajuste fino)


## Cell 0: Mount Google Drive

**Prioridad:** bloqueante. Sin esto no hay datos/caché persistente.

**Qué hacer:**
1. Montar Drive.
2. Verificar que existe `/content/drive/MyDrive/GCED/hackudc`.
3. Confirmar que `data/` contiene los CSV.

Si falla esta celda, no continúes con el resto.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Verify the project folder exists
PROJECT_DIR = Path("/content/drive/MyDrive/GCED/hackudc")
if not PROJECT_DIR.exists():
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Created: {PROJECT_DIR}")
else:
    print(f"Found: {PROJECT_DIR}")
    print("Contents:", [p.name for p in PROJECT_DIR.iterdir()])


## Cell 1: Setup

In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip("fashion-clip", "transformers>=4.35.0", "accelerate", "tqdm", "Pillow",
     "requests", "open_clip_torch")

try:
    _pip("faiss-gpu")
except Exception:
    try:
        _pip("faiss-cpu")
    except Exception:
        pass


## Cell 2: Imports & Configuration

## Orden de trabajo recomendado (impacto vs coste)

**Fase A — Crítica (hacer siempre):**
- Datos limpios y descargas completas.
- Embeddings de producto consistentes (sin caché obsoleta).
- Segmentación y crops correctos (min_frac, padding, square pad).

**Fase B — Alta (hacer después):**
- Aumentar cobertura de candidatos (`K_PER_SEGMENT`).
- Ajustar agregación (`SEGMENT_WEIGHTS`, `CROSS_SEG_BONUS`, `WHOLE_IMG_WEIGHT`).

**Fase C — Media/Baja (último paso):**
- PRF (`PRF_TOP_K`, `PRF_WEIGHT`).
- Text re-ranking (`TEXT_RERANK_ALPHA`, `TEXT_RERANK_TOP_N`).

**Regla práctica:** no optimizar Fase C si Fase A todavía no está estable.

In [ ]:
import os, gc, json, math, time, random, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

import numpy as np
import pandas as pd
import requests
import torch
try:
    import faiss
except ModuleNotFoundError:
    raise ModuleNotFoundError("faiss not found. Re-run Cell 1.")
from PIL import Image
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────────────────────
DRIVE_ROOT = Path("/content/drive/MyDrive/GCED/hackudc")
DATA_DIR   = DRIVE_ROOT / "data"
WORK_DIR   = DRIVE_ROOT
IMG_DIR    = WORK_DIR / "images"
PROD_DIR   = IMG_DIR / "products"
BUND_DIR   = IMG_DIR / "bundles"
SUBM_FILE  = WORK_DIR / "submission.csv"
for d in [DATA_DIR, PROD_DIR, BUND_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hardware ─────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Model ────────────────────────────────────────────────────────────────────
USE_MARQO      = True
MARQO_MODEL_ID = "marqo/marqo-fashionSigLIP"
MODEL_TAG      = "marqo" if USE_MARQO else "fclip"
EMB_FILE       = WORK_DIR / f"product_embeddings_{MODEL_TAG}.npy"
IDS_FILE       = WORK_DIR / f"product_ids_{MODEL_TAG}.json"
FORCE_RECOMPUTE = False

# ── Hyper-params ─────────────────────────────────────────────────────────────
TOP_K            = 15
EMBED_BATCH      = 128
DOWNLOAD_WORKERS = 32
IMG_SIZE         = 224
DOWNLOAD_TIMEOUT = 20
DOWNLOAD_RETRIES = 4
K_PER_SEGMENT    = 400   # candidates per segment (↑ from 200)
SEG_MIN_CROPS    = 1
WHOLE_IMG_WEIGHT = 0.3   # Z-score normalised whole-image bonus
TTA_N_VIEWS      = 3     # orig + flip + centre crop

# Text re-ranking (linear decay)
ENABLE_TEXT_RERANK  = True
TEXT_RERANK_ALPHA   = 0.10
TEXT_RERANK_TOP_N   = 100   # re-rank window (↑ from 60)

# Text-visual ensemble: DISABLED — coarse labels add noise
ENABLE_TEXT_ENSEMBLE = False
# Section boost: DISABLED — causes train/test leakage
ENABLE_SECTION_BOOST = False

# ── Segment weighting (Phase 3) ──────────────────────────────────────────────
SEGMENT_WEIGHTS = {
    "upper_body": 1.2,
    "lower_body": 1.2,
    "dress":      1.0,
    "shoes":      0.9,
    "bag":        0.7,
    "hat":        0.6,
    "scarf_belt": 0.5,
}
CROSS_SEG_BONUS = 0.5  # per extra segment a product appears in

# ── Pseudo-Relevance Feedback (Phase 4) ──────────────────────────────────────
ENABLE_PRF = True
PRF_TOP_K  = 3    # top products averaged for re-query
PRF_WEIGHT = 0.4  # weight of PRF expansion

# ── Soft category filtering (Phase 5) ────────────────────────────────────────
SOFT_CATEGORY_PENALTY = 0.3  # multiply score for out-of-category products

DOWNLOAD_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zara.com/",
    "Sec-Fetch-Mode": "no-cors",
}

print("Configuration loaded.")


In [ ]:
# Ajuste centralizado de prioridades (ejecutar después de la Cell 2)
EXECUTION_PROFILE = "quality"  # "quality" | "fast"

if EXECUTION_PROFILE == "fast":
    # Mantiene núcleo fuerte (embeddings + segmentación), recorta refinamientos
    K_PER_SEGMENT      = 200
    TEXT_RERANK_TOP_N  = 60
    ENABLE_PRF         = False
    TTA_N_VIEWS        = 2
elif EXECUTION_PROFILE == "quality":
    K_PER_SEGMENT      = 400
    TEXT_RERANK_TOP_N  = 100
    ENABLE_PRF         = True
    TTA_N_VIEWS        = 3
else:
    raise ValueError("EXECUTION_PROFILE debe ser 'quality' o 'fast'")

print(f"Perfil: {EXECUTION_PROFILE}")
print(f"K_PER_SEGMENT={K_PER_SEGMENT} | TTA_N_VIEWS={TTA_N_VIEWS} | ENABLE_PRF={ENABLE_PRF} | TEXT_RERANK_TOP_N={TEXT_RERANK_TOP_N}")

## Perfil de ejecución (priorizado)

Usa este selector para alinear esfuerzo con impacto real.
- `"quality"`: máxima calidad (recomendado para entrega final).
- `"fast"`: validación rápida (itera en menos tiempo).

## Cell 3: Data Loading

In [ ]:
bundles_df  = pd.read_csv(DATA_DIR / "bundles_dataset.csv")
products_df = pd.read_csv(DATA_DIR / "product_dataset.csv")
train_df    = pd.read_csv(DATA_DIR / "bundles_product_match_train.csv")
test_df     = pd.read_csv(DATA_DIR / "bundles_product_match_test.csv")

print("=== Dataset Statistics ===")
print(f"Bundles total  : {len(bundles_df):,}")
print(f"Products total : {len(products_df):,}")
print(f"Train pairs    : {len(train_df):,}")

# Ground-truth lookup
train_gt: dict[str, set] = (
    train_df.dropna(subset=["product_asset_id"])
    .groupby("bundle_asset_id")["product_asset_id"]
    .apply(set).to_dict()
)
counts = [len(v) for v in train_gt.values()]
print(f"Train bundles  : {len(train_gt):,}")
print(f"Avg GT/bundle  : {np.mean(counts):.2f}  Max: {max(counts)}")

# URL / description / section lookups
bundle_url:   dict[str, str] = dict(zip(bundles_df["bundle_asset_id"], bundles_df["bundle_image_url"]))
product_url:  dict[str, str] = dict(zip(products_df["product_asset_id"], products_df["product_image_url"]))
product_desc: dict[str, str] = dict(zip(products_df["product_asset_id"], products_df["product_description"].fillna("")))

# Section info
bundle_section_map:  dict[str, int] = {}
product_section_map: dict[str, int] = {}
if "bundle_id_section" in bundles_df.columns:
    bundle_section_map = {row.bundle_asset_id: int(row.bundle_id_section)
                          for row in bundles_df.itertuples() if pd.notna(row.bundle_id_section)}
# Infer product sections from training pairs
if "bundle_id_section" in train_df.columns or "bundle_id_section" in bundles_df.columns:
    merged = train_df.merge(bundles_df[["bundle_asset_id","bundle_id_section"]],
                            on="bundle_asset_id", how="left")
    for row in merged.itertuples():
        if pd.notna(row.bundle_id_section) and row.product_asset_id not in product_section_map:
            product_section_map[row.product_asset_id] = int(row.bundle_id_section)

test_bundle_ids = test_df["bundle_asset_id"].dropna().unique().tolist()
print(f"Test bundles   : {len(test_bundle_ids)}")
print(f"Section map    : {len(bundle_section_map)} bundles, {len(product_section_map)} products")


## Cell 4: Image Download

In [ ]:
def download_image(asset_id: str, url: str, out_dir: Path,
                   timeout: int = DOWNLOAD_TIMEOUT, retries: int = DOWNLOAD_RETRIES) -> bool:
    """Stream-download with PIL verify + temp-file rename. Returns True on success."""
    out_path = out_dir / f"{asset_id}.jpg"
    if out_path.exists() and out_path.stat().st_size > 500:
        return True
    tmp_path = out_path.with_suffix(".tmp")
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=timeout, headers=DOWNLOAD_HEADERS, stream=True)
            r.raise_for_status()
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(8192):
                    if chunk:
                        f.write(chunk)
            # Verify image is valid before committing
            img = Image.open(tmp_path)
            img.verify()
            tmp_path.rename(out_path)
            return True
        except requests.exceptions.HTTPError as e:
            code = e.response.status_code if e.response else 0
            if code in (403, 404, 410):
                break  # permanent failure
            if attempt < retries - 1:
                time.sleep(1.5 ** attempt)
        except Exception:
            if attempt < retries - 1:
                time.sleep(1.5 ** attempt)
        finally:
            if tmp_path.exists():
                tmp_path.unlink(missing_ok=True)
    return False


def batch_download(id_url_pairs, out_dir, desc="Downloading", timeout=DOWNLOAD_TIMEOUT):
    ok_ids, failed_pairs = [], []
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = {pool.submit(download_image, aid, url, out_dir, timeout): (aid, url)
                   for aid, url in id_url_pairs}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
            aid, url = futures[fut]
            (ok_ids if fut.result() else failed_pairs).append((aid, url) if not fut.result() else aid)
    return ok_ids, failed_pairs


def load_image(asset_id: str, img_dir: Path) -> Image.Image | None:
    p = img_dir / f"{asset_id}.jpg"
    if not p.exists():
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None


# Products
product_pairs = list(product_url.items())
_prod_ok, _prod_fail = batch_download(product_pairs, PROD_DIR, "Products pass-1")
if _prod_fail:
    _prod_ok2, _ = batch_download(_prod_fail, PROD_DIR, "Products pass-2", timeout=60)
    _prod_ok += _prod_ok2
valid_product_ids = _prod_ok
print(f"Products: {len(valid_product_ids):,}/{len(product_pairs):,}")

# Bundles
bundle_pairs = list(bundle_url.items())
_bund_ok, _bund_fail = batch_download(bundle_pairs, BUND_DIR, "Bundles  pass-1")
if _bund_fail:
    _bund_ok2, _ = batch_download(_bund_fail, BUND_DIR, "Bundles  pass-2", timeout=60)
    _bund_ok += _bund_ok2
valid_bundle_ids = _bund_ok
print(f"Bundles : {len(valid_bundle_ids):,}/{len(bundle_pairs):,}")

valid_product_set = set(valid_product_ids)
valid_bundle_set  = set(valid_bundle_ids)


## Cell 5: Marqo-FashionSigLIP Loading

In [ ]:
import open_clip

print(f"Loading Marqo-FashionSigLIP (hf-hub:{MARQO_MODEL_ID}) …")
marqo_model, _, marqo_preprocess = open_clip.create_model_and_transforms(
    f"hf-hub:{MARQO_MODEL_ID}"
)
marqo_tokenizer = open_clip.get_tokenizer(f"hf-hub:{MARQO_MODEL_ID}")
marqo_model.to(DEVICE).eval()
print("Marqo-FashionSigLIP loaded.")


def _l2_norm(arr: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(arr, axis=1, keepdims=True).clip(min=1e-8)
    return arr / norms


def _marqo_encode_images_raw(imgs: list) -> np.ndarray:
    tensors = torch.stack([marqo_preprocess(img) for img in imgs]).to(DEVICE)
    with torch.no_grad(), torch.autocast(
            device_type="cuda" if DEVICE == "cuda" else "cpu", dtype=torch.float16):
        out = marqo_model.encode_image(tensors)
    return out.cpu().float().numpy()


def _marqo_encode_texts_raw(texts: list[str]) -> np.ndarray:
    tokens = marqo_tokenizer(texts).to(DEVICE)
    with torch.no_grad():
        out = marqo_model.encode_text(tokens)
    return out.cpu().float().numpy()


def _load_image(path) -> Image.Image:
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return Image.new("RGB", (IMG_SIZE, IMG_SIZE))


def _embed_images(image_paths: list, batch_size: int = EMBED_BATCH) -> np.ndarray:
    """Float32 (N, D) L2-normalised embeddings — parallel I/O, fp16 autocast."""
    all_embs = []
    with ThreadPoolExecutor(max_workers=8) as io_pool:
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Embedding", leave=False):
            batch_paths = image_paths[i: i + batch_size]
            imgs = list(io_pool.map(_load_image, batch_paths))
            all_embs.append(_l2_norm(_marqo_encode_images_raw(imgs)))
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(all_embs, axis=0)


def tta_encode(img: Image.Image, n_views: int = TTA_N_VIEWS) -> np.ndarray:
    """Average embedding over n_views augments. Returns (1, D) L2-normalised."""
    augments = [img]
    if n_views >= 2:
        augments.append(img.transpose(Image.FLIP_LEFT_RIGHT))
    if n_views >= 3:
        w, h = img.size
        m = max(1, int(min(w, h) * 0.05))
        augments.append(img.crop((m, m, w - m, h - m)).resize((w, h), Image.BILINEAR))
    embs = np.stack([_l2_norm(_marqo_encode_images_raw([a]))[0] for a in augments[:n_views]])
    return _l2_norm(embs.mean(axis=0, keepdims=True))


print("Encode helpers + TTA ready.")


## Cell 6: Product Embeddings + FAISS IndexFlatIP
Exact inner-product search — 100% recall at index level, fits in VRAM (~56MB for 27K×512).


In [ ]:
def _ensure_valid_sets():
    global valid_product_set, valid_product_ids, valid_bundle_set, valid_bundle_ids
    prod_on_disk = {p.stem for p in PROD_DIR.glob("*.jpg")}
    bund_on_disk = {p.stem for p in BUND_DIR.glob("*.jpg")}
    if not valid_product_set and prod_on_disk:
        valid_product_set = prod_on_disk
        valid_product_ids = list(prod_on_disk)
        print(f"Rebuilt valid_product_set from disk: {len(valid_product_set):,}")
    if not valid_bundle_set and bund_on_disk:
        valid_bundle_set = bund_on_disk
        valid_bundle_ids = list(bund_on_disk)

_ensure_valid_sets()

# Cache validity
if FORCE_RECOMPUTE:
    EMB_FILE.unlink(missing_ok=True); IDS_FILE.unlink(missing_ok=True)

_cache_valid = False
if EMB_FILE.exists() and IDS_FILE.exists():
    with open(IDS_FILE) as f:
        _cached_ids = json.load(f)
    _missing = valid_product_set - set(_cached_ids)
    if _missing:
        print(f"[CACHE STALE] {len(_missing):,} products missing — recomputing …")
        EMB_FILE.unlink(missing_ok=True); IDS_FILE.unlink(missing_ok=True)
    else:
        _cache_valid = True

if _cache_valid:
    print("Loading cached product embeddings …")
    product_embeddings  = np.load(EMB_FILE)
    indexed_product_ids = json.load(open(IDS_FILE))
    print(f"  Loaded {product_embeddings.shape}")
else:
    print("Computing product embeddings (~15-20 min on T4) …")
    ordered_ids   = sorted(valid_product_set)
    ordered_paths = [PROD_DIR / f"{pid}.jpg" for pid in ordered_ids]
    product_embeddings  = _embed_images(ordered_paths, batch_size=EMBED_BATCH)
    indexed_product_ids = ordered_ids
    np.save(EMB_FILE, product_embeddings)
    json.dump(indexed_product_ids, open(IDS_FILE, "w"))
    print(f"  Saved: {product_embeddings.shape}")

idx_to_pid = {i: pid for i, pid in enumerate(indexed_product_ids)}
DIM = product_embeddings.shape[1]

# ── Build FAISS IndexFlatIP (exact search, 100% recall) ──────────────────────
print(f"Building FAISS IndexFlatIP (dim={DIM}, n={len(indexed_product_ids):,}) …")
index_cpu = faiss.IndexFlatIP(DIM)
index_cpu.add(product_embeddings.astype(np.float32))

if DEVICE == "cuda":
    try:
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
        print("  Transferred index to GPU.")
    except Exception as e:
        print(f"  [WARN] GPU index failed ({e}), using CPU.")
        index = index_cpu
else:
    index = index_cpu
    print("  Using CPU index.")

print(f"  FAISS ntotal = {index.ntotal:,}")
if index.ntotal < 1000:
    raise RuntimeError(f"Only {index.ntotal} products indexed — something went wrong.")
elif index.ntotal < 20_000:
    print(f"  [WARN] Only {index.ntotal:,} products (expected ~27K). Downloads may have failed.")


## Cell 8: SegFormer B2 Clothes Segmentation

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch.nn.functional as F

print("Loading SegFormer B2 clothes segmentation model …")
SEG_MODEL_ID = "mattmdjaga/segformer_b2_clothes"
seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID)
seg_model     = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID)
seg_model.to(DEVICE).eval()
print("SegFormer loaded.")

ATR_LABELS = {0:"Background",1:"Hat",2:"Hair",3:"Sunglasses",4:"Upper-clothes",
              5:"Skirt",6:"Pants",7:"Dress",8:"Belt",9:"Left-shoe",10:"Right-shoe",
              11:"Face",12:"Left-leg",13:"Right-leg",14:"Left-arm",15:"Right-arm",
              16:"Bag",17:"Scarf"}

SEGMENT_GROUPS = {
    "upper_body": [4],
    "lower_body":  [5, 6],
    "dress":       [7],
    "shoes":       [9, 10],
    "bag":         [16],
    "hat":         [1],
    "scarf_belt":  [8, 17],
}

SEGMENT_TO_CATEGORIES = {
    "upper_body": {
        "T-SHIRT","SHIRT","SWEATER","WIND-JACKET","TOPS AND OTHERS","BLAZER","SWEATSHIRT",
        "BABY T-SHIRT","POLO SHIRT","CARDIGAN","WAISTCOAT","OVERSHIRT","BODYSUIT","COAT",
        "SWIMSUIT","BABY SHIRT","BABY JACKET/COAT","BABY SWEATER","TRENCH RAINCOAT",
        "NIGHTIE/PYJAMAS","BABY TRACKSUIT","BABY CARDIGAN","BABY SWIMSUIT","BABY WIND-JACKET",
        "3/4 COAT","SLEEVELESS PAD. JACKET","KNITTED WAISTCOAT","BABY WAISTCOAT",
        "BABY POLO SHIRT","ANORAK","PARKA","BABY BODY","BABY PYJAMA","BATHROBE/DRES.GOWN",
        "LEISURE AND SPORTS","NEWBORN","NEWBORN TRICOT",
    },
    "lower_body": {
        "TROUSERS","SKIRT","BERMUDA","BABY TROUSERS","SHORTS","LEGGINGS","BABY SKIRT",
        "BABY BERMUDAS","BABY LEGGINGS","STOCKINGS-TIGHTS",
    },
    "dress": {
        "DRESS","OVERALL","BABY DRESS","BABY OVERALL","BABY ROMPER SUIT","BIB OVERALL",
        "BABY OUTFIT","ENSEMBLE..SET","UNIFORM",
    },
    "shoes": {
        "SHOES","BOOT","SANDAL","SPORT SHOES","FLAT SHOES","HEELED SHOES","TRAINERS",
        "ANKLE BOOT","MOCCASINS","RUNNING SHOES","HEELED ANKLE BOOT","HEELED BOOT",
        "HIGH TOPS","FLAT BOOT","FLAT ANKLE BOOT","ATHLETIC FOOTWEAR","HOME SHOES",
        "BEACH SANDAL","RAIN BOOT","WEDGE","SPORTY SANDAL","SOCKS","BABY SOCKS","VAMP/PINKY",
    },
    "bag":        {"HAND BAG-RUCKSACK","PURSE WALLET","WALLETS"},
    "hat":        {"HAT","BABY BONNET"},
    "scarf_belt": {"SCARF","BELT","TIE","GLOVES","BOW TIE/CUMMERBAND","SHAWL/FOULARD","SUSPENDERS"},
}


@torch.no_grad()
def segment_image(pil_img: Image.Image) -> np.ndarray:
    inputs  = seg_processor(images=pil_img, return_tensors="pt").to(DEVICE)
    logits  = seg_model(**inputs).logits
    orig_h, orig_w = pil_img.height, pil_img.width
    up = F.interpolate(logits, size=(orig_h, orig_w), mode="bilinear", align_corners=False)
    return up.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.int32)


def _square_pad(img: Image.Image) -> Image.Image:
    """Pad image to square with white background.
    Marqo-FashionSigLIP was trained on square product images;
    padding instead of stretching preserves aspect ratio."""
    w, h = img.size
    if w == h:
        return img
    side = max(w, h)
    result = Image.new("RGB", (side, side), (255, 255, 255))
    result.paste(img, ((side - w) // 2, (side - h) // 2))
    return result


def extract_segment_crops(pil_img: Image.Image, seg_map: np.ndarray,
                          min_frac: float = 0.03, padding: float = 0.10) -> dict[str, Image.Image]:
    """Extract bounding-box crops per segment group, square-padded.
    min_frac=0.03 (↓ from 0.05) detects smaller accessories.
    padding=0.10 (↑ from 0.05) includes more context."""
    W, H = pil_img.size
    total = H * W
    crops = {}
    for seg_name, label_ids in SEGMENT_GROUPS.items():
        mask = np.isin(seg_map, label_ids)
        if mask.sum() < min_frac * total:
            continue
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        y0, y1 = int(rows.min()), int(rows.max())
        x0, x1 = int(cols.min()), int(cols.max())
        py = int((y1 - y0 + 1) * padding)
        px = int((x1 - x0 + 1) * padding)
        crop = pil_img.crop((max(0, x0-px), max(0, y0-py),
                             min(W, x1+px+1), min(H, y1+py+1)))
        if crop.width > 5 and crop.height > 5:
            crops[seg_name] = _square_pad(crop)  # square-pad for SigLIP
    return crops


def _category_match(desc: str, allowed: set) -> bool:
    if not desc.strip():
        return True
    d = desc.strip().upper()
    for cat in allowed:
        c = cat.upper()
        if d == c or d.startswith(c) or c.startswith(d) or c in d or d in c:
            return True
    return False

print("SegFormer helpers ready.")


## Cell 9: Improved Retrieval Pipeline (Phases 0-5)

1. SegFormer → square-padded crops (10% padding, 3% min area)
2. TTA encode (3 views) → FAISS exact search (K=400/segment)
3. Soft category filter (0.3× penalty, not hard exclusion)
4. Z-score normalisation → segment-weighted sum
5. Cross-segment intersection bonus
6. Whole-image Z-score bonus
7. Pseudo-Relevance Feedback (top-3 product re-query)
8. Text re-ranking (linear decay, α=0.10, top-100)


In [ ]:
def text_rerank(candidate_pids: list[str], crop_embs: np.ndarray,
                alpha: float = TEXT_RERANK_ALPHA) -> list[str]:
    """Linear combination of image rank (linear decay) + text cross-similarity."""
    if not candidate_pids:
        return candidate_pids
    descriptions = [product_desc.get(pid, "") for pid in candidate_pids]
    non_empty    = [(i, d) for i, d in enumerate(descriptions) if d.strip()]
    if not non_empty:
        return candidate_pids
    indices_ne, descs_ne = zip(*non_empty)
    text_embs      = _l2_norm(_marqo_encode_texts_raw(list(descs_ne)))
    sim            = crop_embs @ text_embs.T
    text_scores_ne = sim.max(axis=0)
    # Linear decay (gentler than reciprocal — doesn't over-penalise ranks 5-15)
    img_rank     = np.array([1.0 - i / len(candidate_pids) for i in range(len(candidate_pids))])
    text_full    = np.zeros(len(candidate_pids))
    for li, oi in enumerate(indices_ne):
        text_full[oi] = float(text_scores_ne[li])
    combined = (1 - alpha) * img_rank + alpha * text_full
    order    = np.argsort(combined)[::-1]
    return [candidate_pids[j] for j in order]


def predict_segmented(bundle_ids: list[str], k: int = TOP_K) -> dict[str, list[str]]:
    """
    Full improved pipeline (Phases 0-5):
    1. SegFormer → square-padded crops (10% padding, min_frac=3%)
    2. TTA encode (3 views) → FAISS IndexFlatIP (K=400/segment)
    3. Soft category filtering (0.3× penalty)
    4. Z-score normalisation → segment-weighted sum
    5. Cross-segment intersection bonus
    6. Whole-image Z-score bonus
    7. Pseudo-Relevance Feedback (PRF re-query)
    8. Text re-ranking (linear decay)
    """
    predictions: dict[str, list[str]] = {}

    for bid in tqdm(bundle_ids, desc="Segmented predict"):
        if bid not in valid_bundle_set:
            continue
        img = load_image(bid, BUND_DIR)
        if img is None:
            continue

        # ── Segmentation ────────────────────────────────────────────────────
        try:
            seg_map = segment_image(img)
            crops   = extract_segment_crops(img, seg_map)
        except Exception as e:
            print(f"  [WARN] Segmentation failed for {bid}: {e}")
            crops = {}

        # ── Adaptive routing ────────────────────────────────────────────────
        if len(crops) < SEG_MIN_CROPS:
            emb = tta_encode(img)
            sc, idx_ = index.search(emb, k)
            predictions[bid] = [idx_to_pid[int(i)] for i in idx_[0] if int(i) in idx_to_pid][:k]
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            continue

        # ── TTA-encode crops ─────────────────────────────────────────────────
        crop_list  = list(crops.values())
        seg_names  = list(crops.keys())
        embs_norm  = np.vstack([tta_encode(c) for c in crop_list])  # (n_crops, D)

        K_dynamic  = K_PER_SEGMENT

        # ── FAISS query per segment ──────────────────────────────────────────
        scores_map: dict[str, float] = {}
        seg_hit_count: dict[str, int] = {}  # product → num segments it appears in

        scores_arr, indices_arr = index.search(embs_norm.astype(np.float32), K_dynamic)

        for seg_name, seg_scores, seg_indices in zip(seg_names, scores_arr, indices_arr):
            allowed    = SEGMENT_TO_CATEGORIES.get(seg_name, set())
            seg_weight = SEGMENT_WEIGHTS.get(seg_name, 1.0)

            candidates: list[tuple[str, float]] = []
            for sc, iv in zip(seg_scores, seg_indices):
                pid = idx_to_pid.get(int(iv))
                if pid is None:
                    continue
                score = float(sc)
                # ── Soft category filtering (Phase 5) ────────────────────────
                desc = product_desc.get(pid, "")
                if allowed and not _category_match(desc, allowed):
                    score *= SOFT_CATEGORY_PENALTY
                candidates.append((pid, score))

            if not candidates:
                continue

            # ── Per-segment Z-score normalisation ───────────────────────────
            raw_vals = np.array([s for _, s in candidates], dtype=np.float32)
            if len(raw_vals) > 1:
                mu, sigma = raw_vals.mean(), raw_vals.std() + 1e-8
                candidates = [(p, float((s - mu) / sigma)) for p, s in candidates]

            # ── Confidence weighting (Phase 3) ──────────────────────────────
            top5_sims = raw_vals[:5] if len(raw_vals) >= 5 else raw_vals
            confidence = max(float(top5_sims.mean()), 0.1) if len(top5_sims) > 0 else 1.0

            # ── Weighted sum aggregation ─────────────────────────────────────
            seen_in_seg = set()
            for pid, sc in candidates:
                scores_map[pid] = scores_map.get(pid, 0.0) + sc * seg_weight * confidence
                if pid not in seen_in_seg:
                    seg_hit_count[pid] = seg_hit_count.get(pid, 0) + 1
                    seen_in_seg.add(pid)

        # ── Cross-segment intersection bonus (Phase 3) ──────────────────────
        for pid, count in seg_hit_count.items():
            if count >= 2:
                scores_map[pid] += CROSS_SEG_BONUS * (count - 1)

        # ── Whole-image embedding bonus (Z-score normalised) ─────────────────
        whole_emb = _l2_norm(_marqo_encode_images_raw([img]))
        w_scores, w_indices = index.search(whole_emb.astype(np.float32), K_PER_SEGMENT)
        w_raw = w_scores[0]
        if len(w_raw) > 1:
            mu_w = w_raw.mean(); sigma_w = w_raw.std() + 1e-8
            w_norm = (w_raw - mu_w) / sigma_w
        else:
            w_norm = w_raw
        for norm_sc, iv in zip(w_norm, w_indices[0]):
            pid = idx_to_pid.get(int(iv))
            if pid:
                scores_map[pid] = scores_map.get(pid, 0.0) + WHOLE_IMG_WEIGHT * float(norm_sc)

        # ── Pseudo-Relevance Feedback (Phase 4) ─────────────────────────────
        if ENABLE_PRF:
            for seg_idx, seg_name in enumerate(seg_names):
                seg_i = indices_arr[seg_idx]
                top_prod_indices = [int(iv) for iv in seg_i[:PRF_TOP_K]
                                    if int(iv) < len(product_embeddings)]
                if not top_prod_indices:
                    continue
                prf_emb = product_embeddings[top_prod_indices].mean(axis=0, keepdims=True)
                prf_emb = _l2_norm(prf_emb)
                prf_scores, prf_indices = index.search(prf_emb.astype(np.float32), K_PER_SEGMENT // 2)
                prf_raw = prf_scores[0]
                if len(prf_raw) > 1:
                    mu_p = prf_raw.mean(); sigma_p = prf_raw.std() + 1e-8
                    prf_norm = (prf_raw - mu_p) / sigma_p
                else:
                    prf_norm = prf_raw
                seg_w = SEGMENT_WEIGHTS.get(seg_name, 1.0)
                for norm_sc, iv in zip(prf_norm, prf_indices[0]):
                    pid = idx_to_pid.get(int(iv))
                    if pid:
                        scores_map[pid] = scores_map.get(pid, 0.0) + PRF_WEIGHT * seg_w * float(norm_sc)

        # ── Text re-ranking ──────────────────────────────────────────────────
        ranked = sorted(scores_map, key=scores_map.get, reverse=True)
        top_pids = ranked[:TEXT_RERANK_TOP_N]
        if ENABLE_TEXT_RERANK and top_pids:
            top_pids = text_rerank(top_pids, embs_norm, alpha=TEXT_RERANK_ALPHA)

        predictions[bid] = top_pids[:k]
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return predictions


_ensure_valid_sets()
print("Running segmented pipeline on test bundles …")
segmented_preds = predict_segmented(test_bundle_ids, k=TOP_K)
print(f"Segmented predictions for {len(segmented_preds)} bundles.")


## Cell 10: Validation (Recall@15)

In [ ]:
def recall_at_k(predictions, ground_truth, k=TOP_K):
    total, n = 0.0, 0
    for bid, gt_set in ground_truth.items():
        if not gt_set:
            continue
        preds = predictions.get(bid, [])[:k]
        total += len(set(preds) & gt_set) / len(gt_set)
        n += 1
    return total / n if n > 0 else 0.0


train_bundle_ids_with_gt = list(train_gt.keys())
random.seed(42)
VAL_SAMPLE = random.sample(train_bundle_ids_with_gt, min(500, len(train_bundle_ids_with_gt)))
val_gt = {b: train_gt[b] for b in VAL_SAMPLE if b in train_gt}

print(f"Validating on {len(VAL_SAMPLE)} sampled train bundles …")
val_preds   = predict_segmented(VAL_SAMPLE)
recall_seg  = recall_at_k(val_preds, val_gt)

print(f"\n{'='*50}")
print(f"Recall@{TOP_K}  Segmented : {recall_seg:.4f}")
print(f"{'='*50}")

# Per-complexity
bins = {
    "easy (1-2)":   [b for b in VAL_SAMPLE if 1 <= len(train_gt.get(b,[])) <= 2],
    "medium (3-5)": [b for b in VAL_SAMPLE if 3 <= len(train_gt.get(b,[])) <= 5],
    "hard (6+)":    [b for b in VAL_SAMPLE if len(train_gt.get(b,[])) >= 6],
}
print("\nPer-complexity Recall@15:")
for label, bids in bins.items():
    sub_gt = {b: train_gt[b] for b in bids if b in train_gt}
    r = recall_at_k(val_preds, sub_gt)
    print(f"  {label:16s}: {r:.4f}  n={len(bids)}")


## Cell 11: Generate Submission

In [ ]:
final_preds = segmented_preds

# Global fallback: most-frequent products in training
product_freq = Counter(
    pid for gt_set in train_gt.values() for pid in gt_set if pid in valid_product_set
)
fallback_pids = [pid for pid, _ in product_freq.most_common(TOP_K)]
print(f"Fallback products (top-{TOP_K} by train frequency): {fallback_pids[:3]} …")

rows = []
missing_count = 0
for bid in test_bundle_ids:
    preds = list(final_preds.get(bid, []))
    if not preds:
        preds = list(fallback_pids)
        missing_count += 1
    # Pad to TOP_K if needed
    for fp in fallback_pids:
        if len(preds) >= TOP_K:
            break
        if fp not in preds:
            preds.append(fp)
    for pid in preds[:TOP_K]:
        rows.append({"bundle_asset_id": bid, "product_asset_id": pid})

submission_df = pd.DataFrame(rows)
submission_df.to_csv(SUBM_FILE, index=False)
print(f"\nSubmission saved: {SUBM_FILE}")
print(f"  Rows: {len(submission_df):,}  |  Fallback bundles: {missing_count}")

# Sanity checks
covered = set(submission_df["bundle_asset_id"].unique())
missing = set(test_bundle_ids) - covered
assert len(missing) == 0, f"Missing test bundles: {missing}"
max_preds = submission_df.groupby("bundle_asset_id")["product_asset_id"].count().max()
assert max_preds <= TOP_K, f"Bundle has {max_preds} predictions (max={TOP_K})"
all_cat  = set(products_df["product_asset_id"])
bad_pids = set(submission_df["product_asset_id"]) - all_cat
if bad_pids:
    print(f"  [WARN] {len(bad_pids)} invalid PIDs — removing")
    submission_df = submission_df[~submission_df["product_asset_id"].isin(bad_pids)]
    submission_df.to_csv(SUBM_FILE, index=False)
print(f"  [OK] All {len(test_bundle_ids)} test bundles present, max {max_preds} preds each.")
print("\n" + submission_df.head(10).to_string(index=False))


## Cell 12: Summary

In [ ]:
print("\n" + "="*60)
print("HACKUDC 2026 — IMPROVED SOLUTION SUMMARY")
print("="*60)
print(f"Products indexed        : {index.ntotal:,}")
print(f"Embedding dimension     : {DIM}")
print(f"FAISS backend           : {'GPU' if DEVICE == 'cuda' else 'CPU'} (IndexFlatIP, exact)")
print(f"Retrieval model         : Marqo-FashionSigLIP (512-dim)")
print(f"Segmentation model      : SegFormer B2 (ATR 17 classes)")
print(f"TTA views               : {TTA_N_VIEWS}")
print(f"Aggregation             : Z-score + segment-weighted sum")
print(f"Segment weights         : {SEGMENT_WEIGHTS}")
print(f"Cross-segment bonus     : {CROSS_SEG_BONUS}")
print(f"Whole-image bonus       : {WHOLE_IMG_WEIGHT} (Z-score normalised)")
print(f"PRF (re-query)          : {ENABLE_PRF} (top-{PRF_TOP_K}, weight={PRF_WEIGHT})")
print(f"Soft category penalty   : {SOFT_CATEGORY_PENALTY}")
print(f"Text re-ranking         : {ENABLE_TEXT_RERANK} (alpha={TEXT_RERANK_ALPHA}, top-{TEXT_RERANK_TOP_N})")
print(f"Validation Recall@15    : {recall_seg:.4f}")
print(f"Submission rows         : {len(submission_df):,}")
print("="*60)
